In [1]:

# ============================================================
# FMR1 LFP PROCESSING
# Angelaki mouse ASD — spontaneous / intertrial wheel-still
# ============================================================

%pip -q install -U ONE-api ibllib iblatlas scipy

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
from one.api import ONE
from iblatlas.regions import BrainRegions
from brainbox.io.one import SpikeSortingLoader
from scipy.signal import welch

import numpy as np
import pandas as pd
import hashlib
import gc

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

br = BrainRegions()

GENOTYPE = "Fmr1"
PROJECT = "angelaki_mouseASD"

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous/Fmr1/processing"
)

ROOT.mkdir(parents=True, exist_ok=True)

INSERTION_DIR = ROOT / "per_insertion"
INSERTION_DIR.mkdir(exist_ok=True)

# Keep the same parameters used for WT.
N_WINDOWS = 30
WINDOW_S = 2.0
GUARD_S = 0.5
TUKEY_K = 3.0
MIN_ANIMALS = 5

BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha-like": (8, 12),
    "Beta": (12, 30),
    "Low gamma": (30, 55),
    "High gamma": (65, 100)
}

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

print("ONE initialized")
print("Genotype:", GENOTYPE)
print("Output:", ROOT)



Mounted at /content/drive
ONE initialized
Genotype: Fmr1
Output: /content/drive/MyDrive/Reanalysis/Our_Analysis/lfp/spontaneous/Fmr1/processing


In [2]:

# ============================================================
# STEP 2 — Fmr1 LF inventory
# ============================================================

metadata_url = (
    "https://raw.githubusercontent.com/"
    "Noel-Betelgeuse/autism-neuropixels-reanalysis/"
    "main/Our_Analysis/Spontaneous/04_results/"
    "WT_Fmr1_mouse_level.csv"
)

ap_data = pd.read_csv(metadata_url)

fmr_subjects = sorted(
    ap_data.loc[
        ap_data["genotype"] == "Fmr1",
        "subject"
    ].dropna().astype(str).unique()
)

if not fmr_subjects:
    raise RuntimeError("No Fmr1 subjects in metadata")

print("Fmr1 subjects:", len(fmr_subjects))
print(fmr_subjects)

inventory_rows = []

for i, subject in enumerate(fmr_subjects, 1):

    eids = one.search(
        project=PROJECT,
        subject=subject
    )

    print(f"[{i}/{len(fmr_subjects)}] {subject}: {len(eids)} sessions")

    for eid_i in eids:
        eid_i = str(eid_i)

        info = one.alyx.rest(
            "sessions", "read", id=eid_i
        )

        if info.get("subject") != subject:
            continue

        insertions = one.alyx.rest(
            "insertions",
            "list",
            session=eid_i
        )

        dataset_paths = [
            str(p) for p in one.list_datasets(eid_i)
        ]

        for ins in insertions:
            probe = str(ins["name"])

            lf_files = [
                p for p in dataset_paths
                if p.startswith(f"raw_ephys_data/{probe}/")
                and p.endswith(".lf.cbin")
            ]

            inventory_rows.append({
                "subject": subject,
                "genotype": GENOTYPE,
                "eid": eid_i,
                "pid": str(ins["id"]),
                "probe": probe,
                "date": info.get("start_time"),
                "lf_available": bool(lf_files),
                "lf_files": len(lf_files)
            })

inventory_all = pd.DataFrame(inventory_rows)

if inventory_all.empty:
    raise RuntimeError("No Fmr1 probe insertions found")

inventory = (
    inventory_all.loc[inventory_all["lf_available"]]
    .drop_duplicates("pid")
    .reset_index(drop=True)
)

inventory_all.to_csv(
    ROOT / "Fmr1_LFP_inventory_all.csv",
    index=False
)

inventory.to_csv(
    ROOT / "Fmr1_LFP_inventory.csv",
    index=False
)

print("\nFMR1 INVENTORY")
print("Animals:", inventory["subject"].nunique())
print("Sessions:", inventory["eid"].nunique())
print("LF insertions:", len(inventory))


Fmr1 subjects: 15
['FMR009', 'FMR010', 'FMR011', 'FMR012', 'FMR014', 'FMR015', 'FMR019', 'FMR020', 'FMR024', 'FMR025', 'FMR027', 'FMR028', 'FMR029', 'FMR031', 'FMR032']


/usr/local/lib/python3.13/dist-packages/one/webclient.py:1053: RuntimeWarning: The shared public account will be retired soon. Register your own account at https://openalyx.internationalbrainlab.org/signup
  emit(message)


[1/15] FMR009: 105 sessions
[2/15] FMR010: 50 sessions
[3/15] FMR011: 79 sessions
[4/15] FMR012: 60 sessions
[5/15] FMR014: 70 sessions
[6/15] FMR015: 42 sessions
[7/15] FMR019: 66 sessions
[8/15] FMR020: 30 sessions
[9/15] FMR024: 35 sessions
[10/15] FMR025: 61 sessions
[11/15] FMR027: 58 sessions
[12/15] FMR028: 38 sessions
[13/15] FMR029: 63 sessions
[14/15] FMR031: 41 sessions
[15/15] FMR032: 35 sessions

FMR1 INVENTORY
Animals: 15
Sessions: 54
LF insertions: 105


In [3]:

# ============================================================
# STEP 3 — Core analysis functions
# ============================================================

def classify_region(acronym):
    a = str(acronym)

    if a in ("DG-mo", "DG-sg", "DG-po"):
        return "DG", a

    for region in REGIONS:
        if a == region:
            return region, a

        if region in ("CA1", "CA3", "SUB", "VISp", "MOs"):
            if a.startswith(region):
                suffix = a[len(region):]
                if suffix and (
                    suffix[0].isdigit() or suffix[0] == "-"
                ):
                    return region, a

    return None, None


def get_channel_groups(eid, probe):
    """Discover and resolve all available anatomical mappings."""

    paths = [str(p) for p in one.list_datasets(eid)]
    prefix = f"alf/{probe}/"

    catalog = defaultdict(set)

    for path in paths:
        if not path.startswith(prefix):
            continue

        parts = path.split("/")
        filename = parts[-1]

        if not filename.startswith(
            ("channels.", "electrodeSites.")
        ):
            continue

        revision = None

        if (
            len(parts) >= 3
            and parts[-2].startswith("#")
            and parts[-2].endswith("#")
        ):
            revision = parts[-2][1:-1]
            collection = "/".join(parts[:-2])
        else:
            collection = "/".join(parts[:-1])

        obj = filename.split(".")[0]
        catalog[(collection, revision, obj)].add(filename)

    def load_attr(collection, revision, obj, attr):
        kwargs = {"collection": collection}
        if revision:
            kwargs["revision"] = revision

        return np.asarray(one.load_dataset(
            eid,
            f"{obj}.{attr}.npy",
            **kwargs
        ))

    candidates = []

    for (col, rev, obj), filenames in catalog.items():
        if f"{obj}.brainLocationIds_ccf_2017.npy" in filenames:
            candidates.append((col, rev, obj, filenames))

    # Prefer channels with explicit raw indices.
    candidates.sort(
        key=lambda x: (
            x[2] != "channels",
            x[1] is None
        )
    )

    recovered = []
    errors = []

    for col, rev, obj, names in candidates:

        try:
            atlas = load_attr(
                col, rev, obj,
                "brainLocationIds_ccf_2017"
            ).ravel().astype(int)

            raw = None
            mapping = None

            if f"{obj}.rawInd.npy" in names:
                raw = load_attr(
                    col, rev, obj, "rawInd"
                ).ravel().astype(int)

                if len(raw) != len(atlas):
                    raise ValueError("Atlas/rawInd mismatch")

                mapping = "rawInd"

            elif (
                obj == "electrodeSites"
                and "electrodeSites.localCoordinates.npy" in names
            ):
                site_xy = load_attr(
                    col, rev, obj, "localCoordinates"
                )

                # Match physical site locations to recorded channels.
                for (cc, rr, oo), other in catalog.items():
                    if oo != "channels":
                        continue

                    if not {
                        "channels.rawInd.npy",
                        "channels.localCoordinates.npy"
                    }.issubset(other):
                        continue

                    try:
                        ch_raw = load_attr(
                            cc, rr, "channels", "rawInd"
                        ).ravel().astype(int)

                        ch_xy = load_attr(
                            cc, rr, "channels", "localCoordinates"
                        )

                        if len(ch_raw) != len(ch_xy):
                            continue

                        def xy_key(x):
                            return tuple(np.round(x[:2], 3))

                        xy_lookup = defaultdict(list)

                        for xy, channel in zip(ch_xy, ch_raw):
                            xy_lookup[xy_key(xy)].append(int(channel))

                        raw_match = []
                        atlas_match = []

                        for xy, aid in zip(site_xy, atlas):
                            matches = xy_lookup.get(xy_key(xy), [])

                            if len(matches) == 1:
                                raw_match.append(matches[0])
                                atlas_match.append(int(aid))

                        if raw_match:
                            raw = np.asarray(raw_match, dtype=int)
                            atlas = np.asarray(atlas_match, dtype=int)
                            mapping = "coordinate_match"
                            break

                    except Exception:
                        continue

            if raw is None:
                raise ValueError("No reliable raw-channel mapping")

            if len(np.unique(raw)) != len(raw):
                raise ValueError("Duplicate raw channel indices")

            valid = (raw >= 0) & (raw < 384)
            raw = raw[valid]
            atlas = atlas[valid]

            acronyms = br.id2acronym(atlas)

            groups = defaultdict(list)

            for channel, acronym in zip(raw, acronyms):
                region, subregion = classify_region(acronym)

                if region is None:
                    continue

                # Both region-level and subregion-level estimates.
                groups[(region, "ALL")].append(int(channel))
                groups[(region, subregion)].append(int(channel))

            if not groups:
                raise ValueError("No target brain regions")

            groups = {
                key: np.unique(value).astype(int)
                for key, value in groups.items()
            }

            n_contacts = len(set(
                ch
                for value in groups.values()
                for ch in value
            ))

            recovered.append({
                "groups": groups,
                "n_contacts": n_contacts,
                "collection": col,
                "revision": rev,
                "mapping": mapping
            })

        except Exception as exc:
            errors.append(str(exc))

    if not recovered:
        raise RuntimeError(
            "Anatomy unavailable: " +
            "; ".join(errors[:3])
        )

    best = max(
        recovered,
        key=lambda x: x["n_contacts"]
    )

    return best["groups"]


def get_quiet_windows(eid):
    """Find wheel-still intertrial intervals and sample 2s windows."""

    trials = pd.DataFrame(one.load_dataset(
        eid,
        "_ibl_trials.table.pqt",
        collection="alf"
    )).sort_values("intervals_0").reset_index(drop=True)

    movements = np.asarray(one.load_dataset(
        eid,
        "_ibl_wheelMoves.intervals.npy",
        collection="alf"
    ), dtype=float).reshape(-1, 2)

    movements = movements[
        np.isfinite(movements).all(axis=1)
    ]

    eligible = []

    for i in range(len(trials) - 1):
        start = float(trials.loc[i, "feedback_times"]) + GUARD_S
        end = float(trials.loc[i + 1, "stimOn_times"]) - GUARD_S

        if not np.isfinite(start) or not np.isfinite(end):
            continue

        if end - start < WINDOW_S:
            continue

        segments = [(start, end)]

        overlapping = movements[
            (movements[:, 0] < end) &
            (movements[:, 1] > start)
        ]

        for m0, m1 in overlapping:
            updated = []

            for a, b in segments:
                if m1 <= a or m0 >= b:
                    updated.append((a, b))
                else:
                    if m0 > a:
                        updated.append((a, float(m0)))
                    if m1 < b:
                        updated.append((float(m1), b))

            segments = updated

        eligible.extend([
            (a, b) for a, b in segments
            if b - a >= WINDOW_S
        ])

    seed = int(
        hashlib.md5(str(eid).encode()).hexdigest()[:8],
        16
    )
    rng = np.random.default_rng(seed)

    windows = []

    for idx in rng.permutation(len(eligible)):
        a, b = eligible[idx]

        t0 = float(rng.uniform(a, b - WINDOW_S))
        windows.append((t0, t0 + WINDOW_S))

        if len(windows) >= N_WINDOWS:
            break

    return np.asarray(
        sorted(windows), dtype=float
    ).reshape(-1, 2), len(eligible)


def tukey_keep(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)

    if not finite.any():
        return np.zeros(len(values), dtype=bool)

    q1, q3 = np.percentile(values[finite], [25, 75])
    threshold = q3 + TUKEY_K * (q3 - q1)

    return finite & (values <= threshold)


def integrate_power(psd, frequencies, low, high):
    mask = (
        (frequencies >= low) &
        (frequencies <= high)
    )

    return np.trapezoid(
        psd[..., mask],
        frequencies[mask],
        axis=-1
    )


def calculate_group_power(psd, rms, frequencies, indices):
    """
    PSD: windows x channels x frequencies
    RMS: windows x channels
    """

    subset_psd = psd[:, indices, :]
    subset_rms = rms[:, indices]

    # Global time-window Tukey QC within this anatomical group.
    window_rms = np.median(subset_rms, axis=1)
    window_keep = tukey_keep(window_rms)

    if not window_keep.any():
        return None

    # Per-channel Tukey QC.
    valid = np.zeros_like(subset_rms, dtype=bool)

    for j in range(subset_rms.shape[1]):
        r = subset_rms[window_keep, j]

        if len(r) >= 4:
            valid[window_keep, j] = tukey_keep(r)
        else:
            valid[window_keep, j] = np.isfinite(r)

    powers = {
        name: integrate_power(
            subset_psd, frequencies, low, high
        )
        for name, (low, high) in BANDS.items()
    }

    total = (
        integrate_power(subset_psd, frequencies, 1, 55)
        + integrate_power(subset_psd, frequencies, 65, 100)
    )

    contact_abs = defaultdict(list)
    contact_rel = defaultdict(list)
    n_good_channels = 0

    for j in range(subset_rms.shape[1]):
        good = valid[:, j]

        if not good.any():
            continue

        n_good_channels += 1

        for name in BANDS:
            abs_value = float(
                np.mean(powers[name][good, j])
            )

            rel_value = float(np.mean(
                100 * powers[name][good, j] /
                np.maximum(total[good, j], 1e-20)
            ))

            contact_abs[name].append(abs_value)
            contact_rel[name].append(rel_value)

    if n_good_channels == 0:
        return None

    result = {
        "n_channels": n_good_channels,
        "n_windows_available": len(psd),
        "n_windows_qc": int(window_keep.sum()),
        "n_channel_windows_qc": int(valid.sum())
    }

    for name in BANDS:
        result[f"{name}_absolute_uv2"] = float(
            np.median(contact_abs[name])
        )

        result[f"{name}_relative_pct"] = float(
            np.median(contact_rel[name])
        )

    return result


print("All Fmr1 processing functions initialized.")


All Fmr1 processing functions initialized.


In [4]:

# ============================================================
# STEP 4 — Process all Fmr1 LF insertions
# ============================================================

session_window_cache = {}
log_rows = []

for number, row in enumerate(inventory.itertuples(index=False), 1):

    subject = str(row.subject)
    eid_i = str(row.eid)
    pid_i = str(row.pid)
    probe = str(row.probe)

    output_path = INSERTION_DIR / f"{pid_i}.csv"

    if output_path.exists():
        print(
            f"[{number}/{len(inventory)}] "
            f"SKIP cached {subject} {probe}"
        )
        continue

    print(
        f"[{number}/{len(inventory)}] "
        f"{subject} — {probe}",
        flush=True
    )

    try:
        # Load anatomical channel mapping.
        groups = get_channel_groups(eid_i, probe)

        # Reuse identical behavioral windows between probes.
        if eid_i not in session_window_cache:
            session_window_cache[eid_i] = (
                get_quiet_windows(eid_i)
            )

        windows, n_eligible = session_window_cache[eid_i]

        if len(windows) == 0:
            raise ValueError("No eligible 2-second quiet windows")

        # Open raw LF stream.
        ssl_i = SpikeSortingLoader(
            pid=pid_i, one=one
        )

        reader = ssl_i.raw_electrophysiology(
            band="lf", stream=True
        )

        fs_i = float(reader.fs)
        n_neural = int(reader.nc - reader.nsync)
        n_samples = int(round(WINDOW_S * fs_i))

        selected_channels = np.unique(
            np.concatenate(list(groups.values()))
        )

        selected_channels = selected_channels[
            (selected_channels >= 0) &
            (selected_channels < n_neural)
        ]

        if not len(selected_channels):
            raise ValueError("No valid mapped LF channels")

        local_lookup = {
            int(ch): j
            for j, ch in enumerate(selected_channels)
        }

        group_indices = {}

        for key, raw_channels in groups.items():
            idx = [
                local_lookup[int(ch)]
                for ch in raw_channels
                if int(ch) in local_lookup
            ]

            if idx:
                group_indices[key] = np.asarray(
                    idx, dtype=int
                )

        psd_list = []
        rms_list = []
        loaded_windows = []
        window_errors = []

        for t0, t1 in windows:
            try:
                probe_t0 = float(
                    ssl_i.timesprobe2times(
                        float(t0), direction="reverse"
                    )
                )

                s0 = int(round(probe_t0 * fs_i))
                s1 = s0 + n_samples

                if s0 < 0:
                    continue

                raw_block = np.asarray(
                    reader[s0:s1, :n_neural]
                )

                if raw_block.shape != (n_samples, n_neural):
                    continue

                signals_uv = (
                    raw_block[:, selected_channels]
                    .astype(np.float64) * 1e6
                )

                if not np.isfinite(signals_uv).all():
                    continue

                frequencies, psd = welch(
                    signals_uv,
                    fs=fs_i,
                    window="hann",
                    nperseg=n_samples,
                    noverlap=0,
                    detrend="linear",
                    scaling="density",
                    axis=0
                )

                freq_mask = frequencies <= 100

                psd_list.append(
                    psd[freq_mask, :].T.astype(np.float32)
                )

                rms_list.append(
                    np.std(signals_uv, axis=0)
                )

                loaded_windows.append((t0, t1))

            except Exception as exc:
                window_errors.append(str(exc)[:150])

        if not psd_list:
            raise ValueError(
                "No LF windows loaded: "
                + "; ".join(window_errors[:3])
            )

        psd_array = np.stack(psd_list, axis=0)
        rms_array = np.stack(rms_list, axis=0)
        freqs_used = frequencies[freq_mask]

        results = []

        for (region, subregion), indices in group_indices.items():

            metrics = calculate_group_power(
                psd_array,
                rms_array,
                freqs_used,
                indices
            )

            if metrics is None:
                continue

            results.append({
                "subject": subject,
                "genotype": GENOTYPE,
                "eid": eid_i,
                "pid": pid_i,
                "probe": probe,
                "region": region,
                "subregion": subregion,
                **metrics
            })

        if not results:
            raise ValueError("No valid anatomical group results")

        pd.DataFrame(results).to_csv(
            output_path, index=False
        )

        pd.DataFrame(
            loaded_windows,
            columns=["start_s", "end_s"]
        ).to_csv(
            INSERTION_DIR / f"{pid_i}_windows.csv",
            index=False
        )

        log_rows.append({
            "pid": pid_i,
            "subject": subject,
            "status": "processed",
            "n_windows": len(loaded_windows),
            "n_regions": len(results),
            "error": ""
        })

        print(
            f"  OK | {len(loaded_windows)} windows | "
            f"{len(results)} anatomical groups"
        )

        del psd_array, rms_array, psd_list, rms_list
        gc.collect()

    except Exception as exc:

        log_rows.append({
            "pid": pid_i,
            "subject": subject,
            "status": "failed",
            "n_windows": 0,
            "n_regions": 0,
            "error": str(exc)
        })

        print("  FAILED:", str(exc)[:200])

    # Save current progress.
    pd.DataFrame(log_rows).to_csv(
        ROOT / "Fmr1_LFP_processing_log.csv",
        index=False
    )

print("\nFmr1 batch processing complete.")


[1/105] FMR009 — probe00


(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/alf/probe00/channels.brainLocationIds_ccf_2017.npy: 100%|██████████| 3.12k/3.12k [00:00<00:00, 18.9kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/alf/probe00/channels.rawInd.npy: 100%|██████████| 3.12k/3.12k [00:00<00:00, 19.3kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/alf/probe00/electrodeSites.brainLocationIds_ccf_2017.npy: 100%|██████████| 3.20k/3.20k [00:00<00:00, 19.7kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/alf/probe00/electrodeSites.localCoordinates.npy: 100%|██████████| 3.20k/3.20k [00:00<00:00, 18.0kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/alf/probe00/channels.localCoordinates.npy: 100%|██████████| 6.06k/6.06k [00:00<00

Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/average_template_25.nrrd Bytes: 32998960


100%|██████████| 31.470260620117188/31.470260620117188 [00:00<00:00, 130.94it/s]


Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/annotation_25.nrrd Bytes: 4035363


100%|██████████| 3.848422050476074/3.848422050476074 [00:00<00:00, 51.04it/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/raw_ephys_data/probe00/_spikeglx_ephysData_g0_t0.imec0.lf.ch: 100%|██████████| 125k/125k [00:00<00:00, 591kB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/FMR009/2021-04-23/001/raw_ephys_data/probe00/_spikeglx_ephysData_g0_t0.imec0.lf.meta: 100%|██████████| 13.9k/13.9k [00:00<00:00, 105kB/s]


... (3,422 outputs left out)

2026-11-18 00:10:28 WARNING  spikeglx.py:840  Meta data doesn't have geometry (snsShankMap/snsGeomMap field), returning defaults


2026-11-18 00:10:29 WARNING  spikeglx.py:840  Meta data doesn't have geometry (snsShankMap/snsGeomMap field), returning defaults


2026-11-18 00:10:31 WARNING  spikeglx.py:840  Meta data doesn't have geometry (snsShankMap/snsGeomMap field), returning defaults


  OK | 30 windows | 2 anatomical groups

Fmr1 batch processing complete.


In [5]:

# ============================================================
# STEP 5 — Final Fmr1 animal-level aggregation
# ============================================================

BAND_COLUMNS = [
    f"{name}_{metric}"
    for name in BANDS
    for metric in ["absolute_uv2", "relative_pct"]
]

result_files = sorted(
    p for p in INSERTION_DIR.glob("*.csv")
    if not p.name.endswith("_windows.csv")
)

if not result_files:
    raise RuntimeError("No processed insertion files found")

insertion_results = pd.concat(
    [pd.read_csv(p) for p in result_files],
    ignore_index=True
)

insertion_results = insertion_results.loc[
    insertion_results["genotype"] == GENOTYPE
].copy()

if insertion_results.empty:
    raise RuntimeError("No Fmr1 results available")

# Retain all insertion-level measurements.
insertion_results.to_csv(
    ROOT / "Fmr1_LFP_insertion_results.csv",
    index=False
)

# One result per animal and anatomical group.
animal_results = (
    insertion_results
    .groupby(
        ["subject", "genotype", "region", "subregion"],
        as_index=False
    )
    .agg(
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        },
        n_sessions=("eid", "nunique"),
        n_insertions=("pid", "nunique"),
        total_channels=("n_channels", "sum")
    )
)

# Independent animal count.
coverage = (
    animal_results
    .groupby(["region", "subregion"], as_index=False)
    .agg(n_animals=("subject", "nunique"))
)

coverage["status"] = np.where(
    coverage["n_animals"] >= MIN_ANIMALS,
    "INCLUDED",
    "INSUFFICIENT_DATA"
)

eligible = coverage.loc[
    coverage["status"] == "INCLUDED",
    ["region", "subregion"]
]

final_results = animal_results.merge(
    eligible,
    on=["region", "subregion"],
    how="inner"
)

summary = (
    final_results
    .groupby(["region", "subregion"], as_index=False)
    .agg(
        n_animals=("subject", "nunique"),
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        }
    )
)

# Save all analysis levels.
animal_results.to_csv(
    ROOT / "Fmr1_LFP_all_animal_results.csv",
    index=False
)

coverage.to_csv(
    ROOT / "Fmr1_LFP_region_eligibility.csv",
    index=False
)

final_results.to_csv(
    ROOT / "Fmr1_LFP_final_animal_results.csv",
    index=False
)

summary.to_csv(
    ROOT / "Fmr1_LFP_group_summary.csv",
    index=False
)

print("=" * 60)
print("FMR1 LFP PROCESSING COMPLETE")
print("=" * 60)

print("Animals analyzed:", animal_results["subject"].nunique())
print("Eligible region/subregion groups:", len(eligible))
print("Final animal-level rows:", len(final_results))

print("\nRegion coverage:")
display(coverage.sort_values(
    ["region", "subregion"]
))

print("\nGroup summary:")
display(summary)

print("\nSaved to:", ROOT)


FMR1 LFP PROCESSING COMPLETE
Animals analyzed: 15
Eligible region/subregion groups: 29
Final animal-level rows: 351

Region coverage:


,region,subregion,n_animals,status
0,APN,ALL,12,INCLUDED
1,APN,APN,12,INCLUDED
2,CA1,ALL,13,INCLUDED
3,CA1,CA1,13,INCLUDED
4,CA3,ALL,12,INCLUDED
5,CA3,CA3,12,INCLUDED
6,CP,ALL,12,INCLUDED
7,CP,CP,12,INCLUDED
8,DG,ALL,15,INCLUDED
9,DG,DG-mo,15,INCLUDED



Group summary:


,region,subregion,n_animals,Delta_absolute_uv2,Delta_relative_pct,Theta_absolute_uv2,Theta_relative_pct,Alpha-like_absolute_uv2,Alpha-like_relative_pct,Beta_absolute_uv2,Beta_relative_pct,Low gamma_absolute_uv2,Low gamma_relative_pct,High gamma_absolute_uv2,High gamma_relative_pct
0,APN,ALL,12,3317.636743,40.623728,2739.056763,33.289735,789.665203,10.025107,796.104801,10.725163,372.480432,4.930731,120.255810,1.544880
1,APN,APN,12,3317.636743,40.623728,2739.056763,33.289735,789.665203,10.025107,796.104801,10.725163,372.480432,4.930731,120.255810,1.544880
2,CA1,ALL,13,3490.091006,36.222879,3264.658702,24.919582,965.633448,9.005369,971.603426,13.886760,503.274300,6.771107,132.131326,2.038659
3,CA1,CA1,13,3490.091006,36.222879,3264.658702,24.919582,965.633448,9.005369,971.603426,13.886760,503.274300,6.771107,132.131326,2.038659
4,CA3,ALL,12,8930.691133,39.680185,6245.587217,23.496833,2300.877127,8.760363,3584.224496,12.774375,1508.732408,7.249265,620.594195,2.804239
5,CA3,CA3,12,8930.691133,39.680185,6245.587217,23.496833,2300.877127,8.760363,3584.224496,12.774375,1508.732408,7.249265,620.594195,2.804239
6,CP,ALL,12,1824.268161,44.329853,1129.987953,24.026126,370.128680,9.372053,512.236662,13.303285,252.543586,5.883012,78.491378,1.836314
7,CP,CP,12,1824.268161,44.329853,1129.987953,24.026126,370.128680,9.372053,512.236662,13.303285,252.543586,5.883012,78.491378,1.836314
8,DG,ALL,15,9646.269909,40.714395,7377.461508,28.045231,2220.336392,9.592119,2121.734297,12.086588,989.322105,6.104226,285.343352,1.919177
9,DG,DG-mo,15,8595.303142,38.334252,5470.249222,28.045231,1506.214612,9.295566,1824.211686,12.265338,751.239063,5.405100,230.082457,1.815390



Saved to: /content/drive/MyDrive/Reanalysis/Our_Analysis/lfp/spontaneous/Fmr1/processing
